# train_cloud_model

Этот ноутбук обучает cloud-модель на XGBoost с SMOTE.

Что делает ноутбук:
- загружает датасет через `build_cloud_dataset`
- делит данные на train / test
- применяет SMOTE только к train
- обучает `XGBClassifier`
- считает метрики
- сохраняет модель и артефакты
- сохраняет test set в CSV


In [40]:
import json
from pathlib import Path

import joblib
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

from imblearn.over_sampling import SMOTE

from preprocess import build_cloud_dataset, save_feature_list, CLOUD_FEATURES

## 1. Пути и конфиг

In [41]:
ARTIFACTS_DIR = Path("training/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "cloud_model.json"
FEATURES_PATH = ARTIFACTS_DIR / "cloud_feature_list.json"
METRICS_PATH = ARTIFACTS_DIR / "cloud_metrics.json"
TEST_CSV_PATH = ARTIFACTS_DIR / "cloud_test_set.csv"
TEST_PREDICTIONS_CSV_PATH = ARTIFACTS_DIR / "cloud_test_predictions.csv"

## 2. Загрузка датасета

In [42]:
X, y = build_cloud_dataset()

print("Dataset shape:", X.shape)
print("\nOriginal class distribution:")
print(y.value_counts())

X.head()

Dataset shape: (10000, 6)

Original class distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
0,0.156250,0.700000,0.830278,0.762923,0.864987,0.000000
1,0.171875,0.700000,0.753727,0.825312,0.849450,0.014560
2,0.156250,0.666667,0.801906,0.880570,0.964257,0.024266
3,0.171875,0.666667,0.767110,0.704100,0.737560,0.033972
4,0.171875,0.700000,0.753727,0.713012,0.733866,0.043679


## 3. Train / test split

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution BEFORE SMOTE:")
print(y_train.value_counts())

Train shape: (8000, 6)
Test shape: (2000, 6)

Train distribution BEFORE SMOTE:
Machine failure
0    7729
1     271
Name: count, dtype: int64


## 4. Сохраняем test set в CSV

In [44]:
test_df = X_test.copy()
test_df["Machine failure"] = y_test.values
test_df.to_csv(TEST_CSV_PATH, index=False)

print("Saved test set to:", TEST_CSV_PATH)
test_df.head()


Saved test set to: training\artifacts\cloud_test_set.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],Machine failure
2997,0.53125,0.300000,0.720002,1.000000,1.000000,0.742538,0
4871,1.00000,0.100000,0.809935,0.714795,0.790564,0.655181,0
3858,0.84375,0.166667,0.834560,0.670232,0.763814,1.000000,0
951,0.00000,0.766667,0.807794,0.638146,0.703925,0.291191,0
6463,0.53125,0.366667,0.726961,1.000000,1.000000,0.495025,0


## 5. SMOTE только на train

In [45]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Train shape AFTER SMOTE:", X_train_resampled.shape)
print("\nTrain distribution AFTER SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


Train shape AFTER SMOTE: (15458, 6)

Train distribution AFTER SMOTE:
Machine failure
0    7729
1    7729
Name: count, dtype: int64


In [46]:
SAFE_FEATURE_MAP = {
    "Air temperature [K]": "air_temperature_k",
    "temp_diff": "temp_diff",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "power_kw": "power_kw",
    "Tool wear [min]": "tool_wear_min",
}

X_train_resampled = X_train_resampled.rename(columns=SAFE_FEATURE_MAP)
X_test = X_test.rename(columns=SAFE_FEATURE_MAP)

## 6. Обучение XGBoost

In [47]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=1.0,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled, y_train_resampled)
print("Cloud model trained successfully")

Cloud model trained successfully


## 7. Предсказания и метрики

In [48]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "model_type": "cloud_binary",
    "model_name": "XGBoost + SMOTE",
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 6),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 6),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 6),
    "f1_score": round(float(f1_score(y_test, y_pred, zero_division=0)), 6),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 6),
    "pr_auc": round(float(average_precision_score(y_test, y_prob)), 6),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "features": list(X_train_resampled.columns),
    "train_size_before_smote": int(len(X_train)),
    "train_size_after_smote": int(len(X_train_resampled)),
    "test_size": int(len(X_test)),
}

print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nMetrics:")
print(json.dumps(metrics, indent=2))

Classification report:
              precision    recall  f1-score   support

           0     0.9941    0.9674    0.9806      1932
           1     0.4750    0.8382    0.6064        68

    accuracy                         0.9630      2000
   macro avg     0.7346    0.9028    0.7935      2000
weighted avg     0.9765    0.9630    0.9679      2000


Metrics:
{
  "model_type": "cloud_binary",
  "model_name": "XGBoost + SMOTE",
  "accuracy": 0.963,
  "precision": 0.475,
  "recall": 0.838235,
  "f1_score": 0.606383,
  "roc_auc": 0.965481,
  "pr_auc": 0.780228,
  "confusion_matrix": [
    [
      1869,
      63
    ],
    [
      11,
      57
    ]
  ],
  "features": [
    "air_temperature_k",
    "temp_diff",
    "rotational_speed_rpm",
    "torque_nm",
    "power_kw",
    "tool_wear_min"
  ],
  "train_size_before_smote": 8000,
  "train_size_after_smote": 15458,
  "test_size": 2000
}


## 8. Сохраняем test predictions в CSV

In [49]:
test_predictions_df = X_test.copy()
test_predictions_df["Machine failure"] = y_test.values
test_predictions_df["predicted_label"] = y_pred
test_predictions_df["predicted_probability"] = y_prob
test_predictions_df.to_csv(TEST_PREDICTIONS_CSV_PATH, index=False)

print("Saved test predictions to:", TEST_PREDICTIONS_CSV_PATH)
test_predictions_df.head()


Saved test predictions to: training\artifacts\cloud_test_predictions.csv


,air_temperature_k,temp_diff,rotational_speed_rpm,torque_nm,power_kw,tool_wear_min,Machine failure,predicted_label,predicted_probability
2997,0.53125,0.300000,0.720002,1.000000,1.000000,0.742538,0,1,0.613734
4871,1.00000,0.100000,0.809935,0.714795,0.790564,0.655181,0,0,0.000825
3858,0.84375,0.166667,0.834560,0.670232,0.763814,1.000000,0,1,0.632345
951,0.00000,0.766667,0.807794,0.638146,0.703925,0.291191,0,0,0.001972
6463,0.53125,0.366667,0.726961,1.000000,1.000000,0.495025,0,0,0.279107


## 9. Важность признаков

In [50]:
feature_importance = pd.DataFrame({
    "feature": X_train_resampled.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

,feature,importance
2,rotational_speed_rpm,0.326685
4,power_kw,0.205725
3,torque_nm,0.192004
5,tool_wear_min,0.165023
1,temp_diff,0.069035
0,air_temperature_k,0.041528


## 10. Сохранение модели и артефактов

In [51]:
joblib.dump(model, MODEL_PATH)
save_feature_list(list(X_train_resampled.columns), FEATURES_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(list(X_train_resampled.columns), f, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved feature list to:", FEATURES_PATH)
print("Saved metrics to:", METRICS_PATH)


Feature list saved to: training\artifacts\cloud_feature_list.json
Saved model to: training\artifacts\cloud_model.json
Saved feature list to: training\artifacts\cloud_feature_list.json
Saved metrics to: training\artifacts\cloud_metrics.json


## 11. Пример одного cloud prediction

In [52]:
sample_input = X_test.iloc[[0]].copy()
sample_risk = float(model.predict_proba(sample_input)[0, 1])
sample_pred = int(model.predict(sample_input)[0])

result = {
    "risk_score": round(sample_risk, 4),
    "prediction": "HIGH_RISK" if sample_risk >= 0.5 else "LOW_RISK",
    "features": sample_input.to_dict(orient="records")[0],
}

print(json.dumps(result, indent=2))


{
  "risk_score": 0.6137,
  "prediction": "HIGH_RISK",
  "features": {
    "air_temperature_k": 0.5312499999999983,
    "temp_diff": 0.3000000000000057,
    "rotational_speed_rpm": 0.7200021412703089,
    "torque_nm": 1.0,
    "power_kw": 1.0,
    "tool_wear_min": 0.7425382188789155
  }
}
